In [1]:
import zipfile
import re
import math
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def preprocess(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[^\w\s]", " ", text.lower())
    return " ".join(token.lemma_ for token in nlp(text) if not token.is_space)

# 1. Načtení článků z Medium a odstranění duplicit
with zipfile.ZipFile('archive.zip', 'r') as z:
    csv_filename = [f for f in z.namelist() if f.endswith('.csv')][0]
    df = pd.read_csv(z.open(csv_filename))

df['full_text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
df = df.drop_duplicates(subset=['title', 'text']).reset_index(drop=True)
M = len(df)
print(f"Počet unikátních článků (M): {M}")

df['clean_text'] = df['full_text'].apply(preprocess)
docs = df['clean_text'].tolist()

# Inverzní index pro TF-IDF ze 4. cvičení
index = defaultdict(list)
corpus_counts = {}
for doc_id, doc_text in enumerate(docs):
    words = doc_text.split()
    counts = Counter(words)
    corpus_counts[doc_id] = counts
    for word in counts:
        index[word].append(doc_id)

def tf_idf(query, index, corpus_counts, M):
    query_words = query.split()
    query_counts = Counter(query_words)
    scores = defaultdict(float)
    for word in set(query_words):
        if word in index:
            df_w = len(index[word])
            idf = math.log((M + 1) / df_w)
            for doc_id in index[word]:
                scores[doc_id] += query_counts[word] * corpus_counts[doc_id][word] * idf
    return scores

# 2. Sklearn TfidfVectorizer a CountVectorizer (pro prostý průměr)
vectorizer = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False)
X = vectorizer.fit_transform(docs)
vocab = vectorizer.get_feature_names_out()
vocab_size = len(vocab)

count_vectorizer = CountVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False, vocabulary=vocab)
X_counts = count_vectorizer.fit_transform(docs)

Počet unikátních článků (M): 230


In [2]:
def load_glove(filepath):
    embeddings_index = {}
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            embeddings_index[word] = np.asarray(values[1:], dtype='float32')
    return embeddings_index

def run_glove_experiment(glove_dim):
    print(f"\n=================================================================")
    print(f"                      GLOVE {glove_dim}d")
    print(f"=================================================================")
    glove_dict = load_glove(f'glove.6B.{glove_dim}d.txt')

    E = np.zeros((vocab_size, glove_dim), dtype=np.float32)
    missing_words = []
    in_glove_mask = np.zeros(vocab_size, dtype=np.float32)

    for i, word in enumerate(vocab):
        if word in glove_dict:
            E[i] = glove_dict[word]
            in_glove_mask[i] = 1.0
        else:
            missing_words.append(word)

    print(f"Chybějící slova v GloVe: {len(missing_words)} / {vocab_size} ({len(missing_words)/vocab_size*100:.2f}%)")
    print(f"Ukázka chybějících slov: {missing_words[:20]}")

    # Vážený průměr: čitatel X @ E, jmenovatel součet vah přítomných slov (X @ in_glove_mask)
    denom_weighted = (X @ in_glove_mask)[:, None]
    denom_weighted[denom_weighted == 0] = 1.0
    doc_vecs_weighted = (X @ E) / denom_weighted

    # Prostý průměr: čitatel X_counts @ E, jmenovatel počet přítomných slov (X_counts @ in_glove_mask)
    denom_plain = (X_counts @ in_glove_mask)[:, None]
    denom_plain[denom_plain == 0] = 1.0
    doc_vecs_plain = (X_counts @ E) / denom_plain

    queries = [
        "coursera vs udacity machine learning",
        "deep learning neural networks tensorflow",
        "python data science tutorial"
    ]

    for query in queries:
        q_clean = preprocess(query)
        
        # 1. TF-IDF (4. cvičení)
        tfidf_dict = tf_idf(q_clean, index, corpus_counts, M)
        scores_ex4 = np.zeros(M)
        for doc_id, sc in tfidf_dict.items():
            scores_ex4[doc_id] = sc

        # 2. Prostý průměr GloVe
        q_counts = count_vectorizer.transform([q_clean])
        q_denom_plain = (q_counts @ in_glove_mask)[0]
        q_vec_plain = (q_counts @ E) / (q_denom_plain if q_denom_plain > 0 else 1.0)
        scores_plain = cosine_similarity(q_vec_plain, doc_vecs_plain).flatten()

        # 3. TF-IDF vážený průměr GloVe
        q_tfidf = vectorizer.transform([q_clean])
        q_denom_w = (q_tfidf @ in_glove_mask)[0]
        q_vec_w = (q_tfidf @ E) / (q_denom_w if q_denom_w > 0 else 1.0)
        scores_weighted = cosine_similarity(q_vec_w, doc_vecs_weighted).flatten()

        methods = {
            "1. TF-IDF (4. cvičení)": scores_ex4,
            f"2. Prostý průměr GloVe ({glove_dim}d)": scores_plain,
            f"3. TF-IDF vážený GloVe ({glove_dim}d)": scores_weighted
        }

        print(f"\n=================== Dotaz: '{query}' ===================")
        for name, scores in methods.items():
            top_indices = np.argsort(scores)[::-1][:5]
            print(f"\n--- {name} ---")
            for rank, idx in enumerate(top_indices, 1):
                print(f"{rank}. [{scores[idx]:.4f}] {df.iloc[idx]['title']}")

run_glove_experiment(100)
run_glove_experiment(300)


                      GLOVE 100d
Chybějící slova v GloVe: 3626 / 16427 (22.07%)
Ukázka chybějících slov: ['00000', '0002which', '00061', '00078', '000assuming', '000s', '000th', '000x', '0040424330518', '00634', '0078876', '01783941', '01x', '0246238639582', '02627v1', '03640v1', '072733877455676', '083', '088', '09665']

=================== Dotaz: 'coursera vs udacity machine learning' ===================

--- 1. TF-IDF (4. cvičení) ---
1. [109.2698] Every single Machine Learning course on the internet, ranked by your reviews
2. [107.5264] Coursera vs Udacity for Machine Learning – Hacker Noon
3. [39.2765] In defense of skepticism about deep learning – Gary Marcus – Medium
4. [23.2677] Machine Learning in a Week – Learning New Stuff – Medium
5. [19.7473] A “weird” introduction to Deep Learning – Towards Data Science

--- 2. Prostý průměr GloVe (100d) ---
1. [0.7343] Machine Learning เรียนอะไร, รู้ไปทําไม – O v e r f i t t e d – Medium
2. [0.5911] 30 Amazing Machine Learning Projects 

### Shrnutí a odpovědi k Úloze 1

* **Chybějící slova v GloVe:** Ze slovníku chybí 22,07 % slov (3626 z 16427), což jsou zejména číselné kódy, specifické identifikátory a novotvary.
* **Našel průměr GloVe článek *Coursera vs Udacity for Machine Learning* a proč?** Ne, průměr GloVe vektorů tento článek v top 5 nenašel, na rozdíl od klasického TF-IDF; specifická slova se v průměru dlouhého textu rozmělnila.
* **Jaké hodnoty kosinové podobnosti vycházejí a proč jsou si tak blízké?** Kosinové podobnosti dosahují vysokých hodnot a liší se jen v setinách kvůli husté povaze vektorového prostoru, kde dokumenty se stejným širším tématem směřují do velmi blízkých oblastí.

In [3]:
def search_tfidf(query, alpha=0.7, top_k=5):
    query_tokens = preprocess(query).split()
    query_counts = Counter(query_tokens)
    
    title_scores = defaultdict(float)
    text_scores = defaultdict(float)

    for word in set(query_tokens):
        if word in wiki_title_index:
            df_w = len(wiki_title_index[word])
            idf = math.log((wiki_M + 1) / df_w)
            for doc_id, c_w_d in wiki_title_index[word].items():
                title_scores[doc_id] += query_counts[word] * c_w_d * idf
                
        if word in wiki_text_index:
            df_w = len(wiki_text_index[word])
            idf = math.log((wiki_M + 1) / df_w)
            for doc_id, c_w_d in wiki_text_index[word].items():
                text_scores[doc_id] += query_counts[word] * c_w_d * idf
                
    final_scores = {}
    for doc_id in set(title_scores.keys()).union(text_scores.keys()):
        final_scores[doc_id] = alpha * title_scores.get(doc_id, 0.0) + (1 - alpha) * text_scores.get(doc_id, 0.0)
        
    return sorted(final_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

In [4]:
import pickle
from datasets import load_dataset

class DummyDataset(dict):
    def __init__(self, *args, **kwargs):
        super().__init__()
    def __setstate__(self, state):
        pass
    def __setattr__(self, key, value):
        pass

class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if "datasets" in module or "pyarrow" in module:
            return DummyDataset
        return super().find_class(module, name)

print("Načítám indexy z 4. cvičení...")
with open("wiki_indices.pkl", "rb") as f:
    wiki_data = SafeUnpickler(f).load()

wiki_title_index = wiki_data["title_index"]
wiki_text_index = wiki_data["text_index"]
wiki_M = wiki_data["doc_count"]

local_dataset = load_dataset("wikimedia/wikipedia", "20231101.simple", split="train")
wiki_titles_raw = local_dataset["title"]

c:\Users\Ruda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Načítám indexy z 4. cvičení...


In [5]:
def search_tfidf_wiki(query, alpha=0.7, top_k=5):
    query_tokens = preprocess(query).split()
    query_counts = Counter(query_tokens)
    
    title_scores = defaultdict(float)
    text_scores = defaultdict(float)

    for word in set(query_tokens):
        if word in wiki_title_index:
            df_w = len(wiki_title_index[word])
            idf = math.log((wiki_M + 1) / df_w)
            for doc_id, c_w_d in wiki_title_index[word].items():
                title_scores[doc_id] += query_counts[word] * c_w_d * idf
                
        if word in wiki_text_index:
            df_w = len(wiki_text_index[word])
            idf = math.log((wiki_M + 1) / df_w)
            for doc_id, c_w_d in wiki_text_index[word].items():
                text_scores[doc_id] += query_counts[word] * c_w_d * idf
                
    final_scores = {}
    for doc_id in set(title_scores.keys()).union(text_scores.keys()):
        final_scores[doc_id] = alpha * title_scores.get(doc_id, 0.0) + (1 - alpha) * text_scores.get(doc_id, 0.0)
        
    return sorted(final_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

In [6]:
from sentence_transformers import SentenceTransformer

print("Načítám wiki_embeddings.npy a model...")
emb = np.load("wiki_embeddings.npy").astype(np.float32)
model = SentenceTransformer("sentence-transformers/multi-qa-MiniLM-L6-cos-v1")

def search_dense(query, top_k=5):
    q_emb = model.encode(query, normalize_embeddings=True)
    scores = emb @ q_emb
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(idx, scores[idx]) for idx in top_indices]

# Test 5 dotazů podle zadání
queries_wiki = [
    "how do plants make food from sunlight",
    "largest animal that ever lived",
    "coursera vs udacity machine learning",
    "albert einstein theory of relativity",
    "capital city of australia"
]

for q in queries_wiki:
    print(f"\n=================== Dotaz: '{q}' ===================")
    
    print("\n--- 1. Neuronová síť (multi-qa-MiniLM) ---")
    for rank, (idx, score) in enumerate(search_dense(q), 1):
        print(f"{rank}. [{score:.4f}] {wiki_titles_raw[idx]}")
        
    print("\n--- 2. TF-IDF (4. cvičení) ---")
    for rank, (idx, score) in enumerate(search_tfidf_wiki(q), 1):
        print(f"{rank}. [{score:.4f}] {wiki_titles_raw[idx]}")

Načítám wiki_embeddings.npy a model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1486.03it/s]



=================== Dotaz: 'how do plants make food from sunlight' ===================

--- 1. Neuronová síť (multi-qa-MiniLM) ---
1. [0.5724] Sunlight
2. [0.5516] Photosynthesis
3. [0.5430] Plant
4. [0.5348] Solar energy
5. [0.5276] Light-dependent reaction

--- 2. TF-IDF (4. cvičení) ---
1. [100.0325] Plant
2. [94.9519] Evolution
3. [94.7041] Judaism
4. [92.2441] List of common misconceptions
5. [83.1574] List of rivers of Brazil

=================== Dotaz: 'largest animal that ever lived' ===================

--- 1. Neuronová síť (multi-qa-MiniLM) ---
1. [0.6464] Largest extinct animals
2. [0.6048] Gigantopithecus
3. [0.5984] Largest Artiodactyls
4. [0.5673] Colossal squid
5. [0.5587] Gigantophis

--- 2. TF-IDF (4. cvičení) ---
1. [105.1547] Evolution
2. [103.8672] List of characters in the Camp Half-Blood series
3. [103.6062] Service animal
4. [83.5427] Animal Farm
5. [81.9738] Uncertainty principle

=================== Dotaz: 'coursera vs udacity machine learning' ===============

### Shrnutí a odpovědi k Úloze 2

* **Kdy je lepší který přístup a proč?** Neuronová síť (`multi-qa-MiniLM-L6-cos-v1`) je výrazně lepší pro sémantické dotazy (např. *how do plants make food from sunlight*), protože chápe význam textu bez nutnosti doslovné shody slov. TF-IDF je naopak lepší pro dotazy obsahující specifické pojmy a jména (např. *Albert Einstein*), kde záleží na přesné lexikální shodě.